# Lecture 6 — Convolutional Neural Networks: beat the dense net on real images

In **HW3** you computed a 1-D convolution *by hand* — sliding a tiny kernel over a list of
numbers and summing the overlap. And in the **Lecture 5 notebook** you trained a
fully-connected (dense) net on Fashion-MNIST and got it to roughly **88%** test accuracy.

Now we let PyTorch's real `nn.Conv2d` do that sliding for you, stack a couple of conv
layers, and train on the GPU. The payoff is the whole point of Lecture 6: on the **exact
same Fashion-MNIST data**, a small CNN **beats that ~88% dense baseline — with far *fewer*
parameters**. Then we step up to colour images with **CIFAR-10**, where the GPU really earns
its keep.

> **Before you start: Runtime → Change runtime type → T4 GPU.** Everything below is
> time-boxed to run in a few minutes on a free Colab T4.

## 1. Setup & GPU check

Standard imports, a fixed seed so your run matches the discussion, and the `device` handle
from Lecture 1. We also define the small helpers this notebook carries on its own —
`accuracy` and `plot_curves` — plus `count_params`, since *counting parameters* is half the
lesson today.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU,")
    print("         otherwise the CIFAR-10 section will be slow.")


def accuracy(logits, labels):
    """Fraction of correct predictions for a batch of (N, C) logits."""
    return (logits.argmax(dim=1) == labels).float().mean().item()


def count_params(model):
    """Number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def plot_curves(history, title=""):
    """Plot train/val loss and val accuracy vs epoch from a list of dict rows."""
    epochs = [h["epoch"] for h in history]
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
    ax_loss.plot(epochs, [h["train_loss"] for h in history], "o-", label="train loss")
    ax_loss.plot(epochs, [h["val_loss"] for h in history], "o-", label="val loss")
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("loss"); ax_loss.legend()
    ax_loss.set_title("Loss")
    ax_acc.plot(epochs, [h["val_acc"] for h in history], "o-", color="tab:green")
    ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("val accuracy")
    ax_acc.set_ylim(0, 1); ax_acc.set_title("Validation accuracy")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## 2. Part A — Fashion-MNIST: the same data as the Lecture 5 dense net

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) is a drop-in replacement
for MNIST: **70,000 greyscale 28×28 images** of clothing in **10 classes** (T-shirt,
trouser, sneaker, bag, …). It's *real* in the ways that matter for this lesson — the classes
genuinely overlap (a pullover, a coat, and a shirt look alike in 28×28 grey), so "just
memorise the pixels" won't generalise. Crucially, it is the **identical dataset** the
Lecture 5 notebook fed to a dense net, which is what makes the head-to-head comparison fair.

We hold out 6k of the 60k training images as a **validation** split (to tune against) and
keep the official 10k **test** split for the single final number we report.

In [ ]:
# torchvision downloads Fashion-MNIST on first run (Colab has internet).
fmnist_tf = transforms.ToTensor()  # -> float tensor in [0, 1], shape (1, 28, 28)

fmnist_train_full = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=fmnist_tf
)
fmnist_test = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=fmnist_tf
)

# 90/10 train/val split (seeded for reproducibility)
n_val = 6_000
n_train = len(fmnist_train_full) - n_val
fmnist_train, fmnist_val = random_split(
    fmnist_train_full, [n_train, n_val],
    generator=torch.Generator().manual_seed(0),
)

FMNIST_CLASSES = fmnist_train_full.classes
print("classes:", FMNIST_CLASSES)
print(f"train={len(fmnist_train)}  val={len(fmnist_val)}  test={len(fmnist_test)}")

fm_train_loader = DataLoader(fmnist_train, batch_size=128, shuffle=True)
fm_val_loader   = DataLoader(fmnist_val,   batch_size=256)
fm_test_loader  = DataLoader(fmnist_test,  batch_size=256)

## 3. Look at the data

Seeing the images *is* the real-dataset experience. Here's a grid of samples with their
labels. Notice how similar some classes look in 28×28 greyscale — that visual overlap is
exactly why classes like *Shirt / Pullover / Coat* turn up as confusions later.

In [ ]:
def show_grid(dataset, classes, n=10, color=False, title=""):
    """Show n random samples from an (image, label) dataset."""
    idx = torch.randperm(len(dataset))[:n]
    fig, axes = plt.subplots(1, n, figsize=(1.4 * n, 1.8))
    for ax, i in zip(axes, idx):
        img, label = dataset[int(i)]
        if color:
            ax.imshow(img.permute(1, 2, 0).numpy())  # (C,H,W) -> (H,W,C)
        else:
            ax.imshow(img.squeeze(0).numpy(), cmap="gray")
        ax.set_title(classes[label], fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


show_grid(fmnist_train, FMNIST_CLASSES, n=10, title="Fashion-MNIST samples")

## 4. Build the CNN

This is the canonical small CNN from the Lecture 6 reading (§6): two **conv → ReLU → pool**
blocks that turn pixels into feature maps, then a **Flatten + Linear** head that turns those
features into 10 class scores.

Trace the shapes as the data flows through — channels grow while the spatial grid shrinks:

```
(N, 1, 28, 28)  --conv1(1->8, 3x3, pad 1)-->  (N, 8, 28, 28)  --maxpool 2-->  (N, 8, 14, 14)
                --conv2(8->16, 3x3, pad 1)-->  (N, 16, 14, 14) --maxpool 2-->  (N, 16, 7, 7)
                --flatten-->  (N, 16*7*7 = 784)  --linear-->  (N, 10)
```

That final `16 * 7 * 7 = 784` is why the head is `Linear(784, 10)` — recompute it whenever
you change a kernel, stride, or pool (or just print the shape, as the reading's "in
practice" tip suggests).

In [ ]:
# 🔧 Your turn — this mirrors what you built from scratch; read it (and try writing
# it yourself) before running.
#
# The conv stack IS the model today. Everything else (the loop, the metrics) you
# already wrote in the Lecture 5 notebook. This is the exact net from the L6 reading.

def make_fmnist_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1),   # 1 channel -> 8 maps,  28x28
        nn.ReLU(),
        nn.MaxPool2d(2),                             # 28x28 -> 14x14
        nn.Conv2d(8, 16, kernel_size=3, padding=1),  # 8 maps   -> 16 maps, 14x14
        nn.ReLU(),
        nn.MaxPool2d(2),                             # 14x14 -> 7x7
        nn.Flatten(),
        nn.Linear(16 * 7 * 7, 10),                   # 784 features -> 10 class logits
    )


cnn = make_fmnist_cnn().to(device)

# Sanity-check the wiring with a dummy batch BEFORE training (the reading's tip:
# read the shape off a dummy tensor instead of trusting your arithmetic).
dummy = torch.zeros(4, 1, 28, 28, device=device)
print("output shape for a batch of 4:", tuple(cnn(dummy).shape))   # expect (4, 10)
print("CNN trainable parameters:", count_params(cnn))

## 5. Train

The same five-step loop you wrote in Lecture 5 — forward, loss, `zero_grad`, `backward`,
`step` — wrapped in an epoch loop with a validation pass. The only new habit is moving the
**model once** and **every batch** onto `device`. We define `train_one_epoch`, `evaluate`,
and a small `fit` driver here so the notebook stays self-contained.

In [ ]:
# 🔧 Your turn — the training spine. Identical in shape to the Lecture 5 notebook;
# the only image-specific part is that batches are already (N, C, H, W) tensors.

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)        # data onto the same device as the model
        logits = model(x)                        # 1. forward
        loss = loss_fn(logits, y)                # 2. loss
        optimizer.zero_grad()                    # 3. clear last batch's grads
        loss.backward()                          # 4. backprop
        optimizer.step()                         # 5. update weights
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()                                 # eval mode: BN/dropout behave for test
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs, lr=1e-3, weight_decay=0.0):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history.append(
            {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc}
        )
        print(f"epoch {epoch}: train_loss={train_loss:.3f}  "
              f"val_loss={val_loss:.3f}  val_acc={val_acc:.2%}")
    return history

In [ ]:
# Train the Fashion-MNIST CNN. 5 epochs is plenty to clear the dense baseline; this
# runs in well under a minute on a T4.
cnn = make_fmnist_cnn().to(device)
fm_history = fit(cnn, fm_train_loader, fm_val_loader, epochs=5, lr=1e-3)
plot_curves(fm_history, title="Fashion-MNIST CNN")

## 6. Evaluate — and beat the dense baseline

Report the single test number, then draw the confusion matrix to see *which* classes the
CNN trips on.

In [ ]:
cnn_test_loss, cnn_test_acc = evaluate(cnn, fm_test_loader, nn.CrossEntropyLoss())
print(f"CNN  Fashion-MNIST test accuracy: {cnn_test_acc:.2%}")

In [ ]:
# Confusion matrix on the test set (no sklearn needed).
@torch.no_grad()
def confusion_matrix(model, loader, n_classes):
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.long)
    for x, y in loader:
        preds = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, preds):
            cm[t, p] += 1
    return cm


cm = confusion_matrix(cnn, fm_test_loader, n_classes=10)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(FMNIST_CLASSES, rotation=90, fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(FMNIST_CLASSES, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Fashion-MNIST confusion matrix (CNN)")
fig.colorbar(im, fraction=0.046)
fig.tight_layout()
plt.show()

### The Lecture 6 §1 headline: CNN vs the dense net, on the *same* data

Time to land the lesson. We rebuild a representative **fully-connected** net of the kind the
Lecture 5 notebook used on this exact dataset — `784 → 256 → 10` with a ReLU — and put the
two side by side on **two axes at once: accuracy *and* parameter count**.

The reading's claim (§1–§3) is that the CNN should be **more accurate** *and* carry **far
fewer parameters**, because weight sharing reuses one small kernel across the whole image
instead of learning a separate weight for every pixel-to-hidden connection. Let's measure it
rather than take it on faith.

In [ ]:
# A dense net of the shape used in the Lecture 5 notebook (~88% test accuracy there).
def make_fmnist_mlp():
    return nn.Sequential(
        nn.Flatten(),                 # (N, 1, 28, 28) -> (N, 784)
        nn.Linear(28 * 28, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    )


mlp = make_fmnist_mlp().to(device)
print("training the dense (FC) baseline for comparison...")
mlp_history = fit(mlp, fm_train_loader, fm_val_loader, epochs=5, lr=1e-3)
mlp_test_loss, mlp_test_acc = evaluate(mlp, fm_test_loader, nn.CrossEntropyLoss())
print(f"\nDense (FC) baseline test accuracy: {mlp_test_acc:.2%}")

In [ ]:
# Head-to-head: accuracy AND parameters.
cnn_params = count_params(cnn)
mlp_params = count_params(mlp)

print(f"{'model':<16}{'test acc':>10}{'params':>12}")
print("-" * 38)
print(f"{'Dense (FC)':<16}{mlp_test_acc:>9.2%}{mlp_params:>12,}")
print(f"{'CNN':<16}{cnn_test_acc:>9.2%}{cnn_params:>12,}")
print("-" * 38)
print(f"accuracy jump (CNN - FC): {cnn_test_acc - mlp_test_acc:+.2%}")
print(f"the CNN uses {mlp_params / cnn_params:.1f}x FEWER parameters "
      f"({cnn_params:,} vs {mlp_params:,})")
print()
print("That is the Lecture 6 lesson in two numbers: weight sharing buys BOTH higher")
print("accuracy AND a smaller model. The dense net spends most of its parameters on one")
print("giant 784x256 first layer; the CNN's conv kernels are 9 weights reused everywhere.")

## 7. Part B — CIFAR-10: stepping up to colour

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) is **60,000 colour 32×32 images** in
10 classes (airplane, cat, ship, truck, …). It's a real jump in difficulty from
Fashion-MNIST: **3 RGB channels** instead of 1, cluttered natural backgrounds, and far more
within-class variation (a "cat" can be any breed, pose, or lighting). This is where a GPU
stops being a convenience and becomes the thing that makes training practical.

Two changes versus Part A:

- **`transforms.Normalize`** — we standardise each RGB channel to roughly mean 0 / std 1,
  the same "scale your features" habit from the Lecture 5 reading, now per-channel. (The
  numbers below are the well-known CIFAR-10 channel statistics.)
- **A slightly bigger CNN** — three conv blocks (channels `3 → 32 → 64 → 128`), because
  colour images need more feature maps than greyscale clothing.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train_full = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=cifar_tf
)
cifar_test = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=cifar_tf
)

n_val = 5_000
cifar_train, cifar_val = random_split(
    cifar_train_full, [len(cifar_train_full) - n_val, n_val],
    generator=torch.Generator().manual_seed(0),
)

CIFAR_CLASSES = cifar_train_full.classes
print("classes:", CIFAR_CLASSES)
print(f"train={len(cifar_train)}  val={len(cifar_val)}  test={len(cifar_test)}")

cf_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True, num_workers=2)
cf_val_loader   = DataLoader(cifar_val,   batch_size=256, num_workers=2)
cf_test_loader  = DataLoader(cifar_test,  batch_size=256, num_workers=2)

### Look at CIFAR-10 — in colour

The same grid helper, now with `color=True`. We *un-normalise* first so the colours look
right (normalisation makes the raw tensors look washed out / out of range).

In [ ]:
# Un-normalise for display: x*std + mean, then clamp to [0, 1].
mean_t = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
std_t  = torch.tensor(CIFAR_STD).view(3, 1, 1)


class Denorm:
    """Tiny dataset wrapper that returns display-ready (un-normalised) images."""
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        img, label = self.ds[i]
        return (img * std_t + mean_t).clamp(0, 1), label


show_grid(Denorm(cifar_train), CIFAR_CLASSES, n=10, color=True, title="CIFAR-10 samples")

### Build the bigger CNN

Three `conv → ReLU → pool` blocks take the spatial grid `32 → 16 → 8 → 4`, growing channels
`3 → 32 → 64 → 128`, then a `Linear` head. We read the flattened size off a dummy tensor
instead of hand-computing it.

In [ ]:
# 🔧 Your turn — the conv stack again, scaled up for 3-channel 32x32 inputs.
def make_cifar_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 3  -> 32,  32x32
        nn.ReLU(),
        nn.MaxPool2d(2),                              # 32 -> 16
        nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 -> 64,  16x16
        nn.ReLU(),
        nn.MaxPool2d(2),                              # 16 -> 8
        nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64 -> 128, 8x8
        nn.ReLU(),
        nn.MaxPool2d(2),                              # 8  -> 4
        nn.Flatten(),
        nn.Linear(128 * 4 * 4, 10),                   # 2048 -> 10 logits
    )


cifar_cnn = make_cifar_cnn().to(device)
dummy = torch.zeros(2, 3, 32, 32, device=device)
print("output shape for a batch of 2:", tuple(cifar_cnn(dummy).shape))  # (2, 10)
print("CIFAR CNN trainable parameters:", count_params(cifar_cnn))

### Train on CIFAR-10

Keep it to **5 epochs** — enough to land well above chance (10%) and into the respectable
range for a small from-scratch CNN, while staying under ~4 minutes on a T4. This is the
moment the GPU pays off: the same loop on CPU would crawl.

In [ ]:
cifar_cnn = make_cifar_cnn().to(device)
cf_history = fit(cifar_cnn, cf_train_loader, cf_val_loader, epochs=5, lr=1e-3)
plot_curves(cf_history, title="CIFAR-10 CNN")

cifar_test_loss, cifar_test_acc = evaluate(cifar_cnn, cf_test_loader, nn.CrossEntropyLoss())
print(f"CIFAR-10 CNN test accuracy: {cifar_test_acc:.2%}")

## 8. Experiment — modify and observe

Two changes to try. Run each, watch the validation accuracy, **and compare parameter
counts** with `count_params`. Fill in the ablation table at the end with *your* numbers.

**Experiment 1 — a deeper net.** You already added a third conv block for CIFAR. Compare it
against a two-block version on the *same* data: does the extra block help val accuracy, and
what does it cost in parameters? (Watch where the parameters live — dropping a conv block
actually *grows* the `Linear` head, because the feature map reaching it is bigger.)

**Experiment 2 — data augmentation.** The Lecture 5 reading called augmentation "the
cheapest way to get more data." Add `RandomHorizontalFlip` + `RandomCrop` to the *training*
transform only (never the val/test transform — you don't augment what you measure on) and
retrain. A flipped/shifted truck is still a truck, so the labels survive.

In [ ]:
# Experiment 1: a TWO-block CIFAR CNN (drop the third conv block) for comparison.
def make_cifar_cnn_2block():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),                              # 32 -> 16
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),                              # 16 -> 8
        nn.Flatten(),
        nn.Linear(64 * 8 * 8, 10),                    # 4096 -> 10
    )


cnn2 = make_cifar_cnn_2block().to(device)
print("2-block params:", count_params(cnn2), " | 3-block params:", count_params(cifar_cnn))
cf_history_2 = fit(cnn2, cf_train_loader, cf_val_loader, epochs=5, lr=1e-3)
_, acc2 = evaluate(cnn2, cf_test_loader, nn.CrossEntropyLoss())
print(f"2-block CIFAR test accuracy: {acc2:.2%}  (vs 3-block {cifar_test_acc:.2%})")

In [ ]:
# Experiment 2: data augmentation on the TRAINING set only.
aug_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),        # a flipped truck is still a truck
    transforms.RandomCrop(32, padding=4),     # small shifts
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train_aug_full = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=aug_tf
)
# Reuse the SAME training indices as the un-augmented split so the comparison is fair
# (val/test stay un-augmented — we never augment what we measure on).
cifar_train_aug = Subset(cifar_train_aug_full, cifar_train.indices)
cf_train_aug_loader = DataLoader(cifar_train_aug, batch_size=128, shuffle=True, num_workers=2)

cifar_cnn_aug = make_cifar_cnn().to(device)
cf_history_aug = fit(cifar_cnn_aug, cf_train_aug_loader, cf_val_loader, epochs=5, lr=1e-3)
_, acc_aug = evaluate(cifar_cnn_aug, cf_test_loader, nn.CrossEntropyLoss())
print(f"augmented CIFAR test accuracy: {acc_aug:.2%}  (vs no-aug {cifar_test_acc:.2%})")

### Beat-the-baseline target + ablation table

**Target:** get your CIFAR-10 CNN test accuracy **above 70%** within 5 epochs. The third
conv block plus augmentation is usually enough; if you have headroom, try a few more epochs
or add `weight_decay=1e-4` to `fit`.

Fill in the table with **your** runs (replace the `?`s):

| Model / change                | Test acc | Params  |
|-------------------------------|----------|---------|
| Fashion-MNIST — Dense (FC)    | ?        | ?       |
| Fashion-MNIST — CNN           | ?        | ?       |
| CIFAR-10 — 2-block CNN        | ?        | ?       |
| CIFAR-10 — 3-block CNN        | ?        | ?       |
| CIFAR-10 — 3-block + augment  | ?        | (same)  |

## 9. Reflect — report *your* results

Answer from the numbers you just produced (not from memory):

1. **CNN vs FC on Fashion-MNIST.** What test accuracy did your CNN reach, and by how many
   points did it beat the dense baseline? How many parameters did each have — i.e. how many
   *times fewer* did the CNN use? Why does weight sharing produce both effects at once
   (reading §1–§3)?
2. **Confusion.** Which two or three Fashion-MNIST classes does your CNN confuse most? Does
   that match how similar they look in the sample grid?
3. **Depth.** Did the third conv block raise CIFAR val/test accuracy over the two-block net?
   What did it cost in parameters — and where did most of those parameters go, the conv
   layers or the `Linear` head?
4. **Augmentation.** What did `RandomHorizontalFlip` + `RandomCrop` do to your CIFAR test
   accuracy and to the **train–val gap** in `plot_curves`? Explain the effect using the
   Lecture 5 idea that augmentation is "more data for free."
5. **Hand-off, closed.** In HW3 you summed a kernel against a patch by hand for a single
   output number; `nn.Conv2d` did that millions of times this session. Where in your CIFAR
   run did the GPU clearly earn its place?